# Steel I-O Table Scenario Analysis Visualization v2

This notebook loads and visualizes the 18 scenario analysis CSV files generated from the scenario analyzer.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

# Plotly default template
import plotly.io as pio
pio.templates.default = "plotly_white"

print("Libraries loaded successfully!")
print("✅ Using Plotly for interactive visualizations")

## Load All 18 CSV Files

In [ ]:
# Define output directory
output_dir = Path('output')

# Dictionary to store all dataframes
data = {}

# Define all 18 files to load
files = [
    # IO Table - Sector 1610
    'scenario_indirect_prod_1610_iotable_2020.csv',
    'scenario_indirect_import_1610_iotable_2020.csv',
    'scenario_value_added_1610_iotable_2020.csv',
    'scenario_jobcoeff_1610_iotable_2020.csv',
    'scenario_directemploycoeff_1610_iotable_2020.csv',
    
    # IO Table - Sector 4506
    'scenario_indirect_prod_4506_iotable_2020.csv',
    'scenario_indirect_import_4506_iotable_2020.csv',
    'scenario_value_added_4506_iotable_2020.csv',
    'scenario_jobcoeff_4506_iotable_2020.csv',
    'scenario_directemploycoeff_4506_iotable_2020.csv',
    
    # Hydrogen Table - H2S
    'scenario_productioncoeff_H2S_hydrogentable_2020.csv',
    'scenario_valueaddedcoeff_H2S_hydrogentable_2020.csv',
    'scenario_jobcoeff_H2S_hydrogentable_2020.csv',
    'scenario_directemploycoeff_H2S_hydrogentable_2020.csv',
    
    # Hydrogen Table - H2T
    'scenario_productioncoeff_H2T_hydrogentable_2020.csv',
    'scenario_valueaddedcoeff_H2T_hydrogentable_2020.csv',
    'scenario_jobcoeff_H2T_hydrogentable_2020.csv',
    'scenario_directemploycoeff_H2T_hydrogentable_2020.csv',
]

# Load all files
print("Loading CSV files...\n")
for file in files:
    file_path = output_dir / file
    if file_path.exists():
        # Create a simplified key name
        key = file.replace('scenario_', '').replace('.csv', '')
        data[key] = pd.read_csv(file_path)
        print(f"✅ Loaded: {file}")
        print(f"   Shape: {data[key].shape}")
    else:
        print(f"❌ File not found: {file}")

print(f"\nTotal files loaded: {len(data)}")

## Inspect Data Structure

In [ ]:
# Display available datasets
print("Available datasets:\n")
for i, key in enumerate(data.keys(), 1):
    print(f"{i}. {key}")

## Interactive Plotly Visualizations

Beautiful, interactive visualizations using Plotly.

In [ ]:
# Example 1: Total impact over time for a specific scenario (using Plotly)
key = 'indirect_prod_4506_iotable_2020'
df = data[key]

# Get year columns and calculate total impact per year
year_cols = [col for col in df.columns if col.isdigit()]
years = [int(y) for y in year_cols]
total_impacts = [df[col].sum() / 1000 for col in year_cols]  # Convert to billion won

# Create interactive Plotly line chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=years,
    y=total_impacts,
    mode='lines+markers',
    name='Total Impact',
    line=dict(width=3, color='#3498db'),
    marker=dict(size=8, color='#3498db'),
    hovertemplate='<b>Year %{x}</b><br>Total Impact: %{y:,.2f} B won<extra></extra>'
))

fig.update_layout(
    title=f'Total Impact Over Time: {key}',
    xaxis_title='Year',
    yaxis_title='Total Impact (Billion Won)',
    height=500,
    hovermode='x unified',
    showlegend=False,
    template='plotly_white'
)

fig.show()

In [ ]:
# Example 2: Top affected sectors for a specific year (using Plotly)
key = 'indirect_prod_1610_iotable_2020'
df = data[key]
year = '2050'  # Latest year

if year in df.columns:
    # Get top 15 sectors by absolute impact
    df_year = df[['Sector_Code', 'Sector_Name', year]].copy()
    df_year['abs_impact'] = df_year[year].abs()
    top_sectors = df_year.nlargest(15, 'abs_impact')
    
    # Create color array based on positive/negative values
    colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_sectors[year]]
    
    # Create interactive Plotly horizontal bar chart
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=top_sectors['Sector_Name'],
        x=top_sectors[year],
        orientation='h',
        marker=dict(color=colors),
        text=top_sectors[year].apply(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Sector Code: ' + top_sectors['Sector_Code'].astype(str) + '<br>Impact: %{x:,.0f} M won<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Top 15 Affected Sectors in {year}: {key}',
        xaxis_title='Impact (Million Won)',
        yaxis_title='',
        height=600,
        showlegend=False,
        template='plotly_white'
    )
    
    fig.show()
else:
    print(f"Year {year} not found in dataset")

## Summary Tables for Years 2030, 2040, 2050

Creating 6 summary tables showing total impacts for key years across all effect types.

In [ ]:
def create_summary_table(sector_name, indirect_prod_key, indirect_import_key, value_added_key, 
                         jobcoeff_key, directemploycoeff_key, years=['2030', '2040', '2050'], 
                         is_hydrogen=False):
    """
    Create a summary table for a specific sector/scenario.
    
    Parameters:
    - sector_name: Name of the sector/scenario for display
    - indirect_prod_key: Key for indirect production data
    - indirect_import_key: Key for indirect import data (can be None for hydrogen)
    - value_added_key: Key for value added data
    - jobcoeff_key: Key for job coefficient data
    - directemploycoeff_key: Key for direct employment coefficient data
    - years: List of years to include in the summary
    - is_hydrogen: True if this is hydrogen data (changes job creation units)
    
    Returns:
    - DataFrame with summary statistics
    """
    
    summary_rows = []
    
    for year in years:
        row = {'Year': year}
        
        # Indirect Production (convert from million won to billion won)
        if indirect_prod_key in data and year in data[indirect_prod_key].columns:
            row['Indirect Production (billion won)'] = data[indirect_prod_key][year].sum() / 1000
        else:
            row['Indirect Production (billion won)'] = 0
        
        # Import (convert from million won to billion won) - N/A for hydrogen
        if indirect_import_key and indirect_import_key in data and year in data[indirect_import_key].columns:
            row['Import (billion won)'] = data[indirect_import_key][year].sum() / 1000
        else:
            row['Import (billion won)'] = 'N/A'
        
        # Value Added (convert from million won to billion won)
        if value_added_key in data and year in data[value_added_key].columns:
            row['Value Added (billion won)'] = data[value_added_key][year].sum() / 1000
        else:
            row['Value Added (billion won)'] = 0
        
        # Job Creation - different units for hydrogen vs IO table
        if is_hydrogen:
            # For hydrogen: job creation effect in million won, convert to billion won
            if jobcoeff_key in data and year in data[jobcoeff_key].columns:
                row['Job Creation (billion won)'] = data[jobcoeff_key][year].sum() / 1000
            else:
                row['Job Creation (billion won)'] = 0
        else:
            # For IO table: person/billion won
            if jobcoeff_key in data and year in data[jobcoeff_key].columns:
                row['Job Creation (person/billion won)'] = data[jobcoeff_key][year].sum()
            else:
                row['Job Creation (person/billion won)'] = 0
        
        # Direct Employment (person/billion won)
        if directemploycoeff_key in data and year in data[directemploycoeff_key].columns:
            row['Direct Employment (person/billion won)'] = data[directemploycoeff_key][year].sum()
        else:
            row['Direct Employment (person/billion won)'] = 0
        
        summary_rows.append(row)
    
    df = pd.DataFrame(summary_rows)
    df = df.set_index('Year')
    
    return df

# Create all 6 summary tables
print("="*80)
print("SUMMARY TABLES FOR YEARS 2030, 2040, 2050")
print("="*80)

# Table 1: Sector 1610
print("\n1. SECTOR 1610 (IO Table)")
print("-"*80)
table_1610 = create_summary_table(
    sector_name="Sector 1610",
    indirect_prod_key="indirect_prod_1610_iotable_2020",
    indirect_import_key="indirect_import_1610_iotable_2020",
    value_added_key="value_added_1610_iotable_2020",
    jobcoeff_key="jobcoeff_1610_iotable_2020",
    directemploycoeff_key="directemploycoeff_1610_iotable_2020",
    is_hydrogen=False
)
display(table_1610)

# Table 2: Sector 4506
print("\n2. SECTOR 4506 (IO Table)")
print("-"*80)
table_4506 = create_summary_table(
    sector_name="Sector 4506",
    indirect_prod_key="indirect_prod_4506_iotable_2020",
    indirect_import_key="indirect_import_4506_iotable_2020",
    value_added_key="value_added_4506_iotable_2020",
    jobcoeff_key="jobcoeff_4506_iotable_2020",
    directemploycoeff_key="directemploycoeff_4506_iotable_2020",
    is_hydrogen=False
)
display(table_4506)

# Table 3: H2S
print("\n3. H2S (Hydrogen Table)")
print("-"*80)
table_h2s = create_summary_table(
    sector_name="H2S",
    indirect_prod_key="productioncoeff_H2S_hydrogentable_2020",
    indirect_import_key=None,  # No import for hydrogen
    value_added_key="valueaddedcoeff_H2S_hydrogentable_2020",
    jobcoeff_key="jobcoeff_H2S_hydrogentable_2020",
    directemploycoeff_key="directemploycoeff_H2S_hydrogentable_2020",
    is_hydrogen=True
)
display(table_h2s)

# Table 4: H2T
print("\n4. H2T (Hydrogen Table)")
print("-"*80)
table_h2t = create_summary_table(
    sector_name="H2T",
    indirect_prod_key="productioncoeff_H2T_hydrogentable_2020",
    indirect_import_key=None,  # No import for hydrogen
    value_added_key="valueaddedcoeff_H2T_hydrogentable_2020",
    jobcoeff_key="jobcoeff_H2T_hydrogentable_2020",
    directemploycoeff_key="directemploycoeff_H2T_hydrogentable_2020",
    is_hydrogen=True
)
display(table_h2t)

# Table 5: Integrated 1610 & 4506
print("\n5. INTEGRATED 1610 & 4506 (IO Table)")
print("-"*80)
years = ['2030', '2040', '2050']
integrated_io_rows = []

for year in years:
    row = {'Year': year}
    
    # Sum indirect production from both sectors
    prod_1610 = data["indirect_prod_1610_iotable_2020"][year].sum() / 1000 if year in data["indirect_prod_1610_iotable_2020"].columns else 0
    prod_4506 = data["indirect_prod_4506_iotable_2020"][year].sum() / 1000 if year in data["indirect_prod_4506_iotable_2020"].columns else 0
    row['Indirect Production (billion won)'] = prod_1610 + prod_4506
    
    # Sum import from both sectors
    import_1610 = data["indirect_import_1610_iotable_2020"][year].sum() / 1000 if year in data["indirect_import_1610_iotable_2020"].columns else 0
    import_4506 = data["indirect_import_4506_iotable_2020"][year].sum() / 1000 if year in data["indirect_import_4506_iotable_2020"].columns else 0
    row['Import (billion won)'] = import_1610 + import_4506
    
    # Sum value added from both sectors
    va_1610 = data["value_added_1610_iotable_2020"][year].sum() / 1000 if year in data["value_added_1610_iotable_2020"].columns else 0
    va_4506 = data["value_added_4506_iotable_2020"][year].sum() / 1000 if year in data["value_added_4506_iotable_2020"].columns else 0
    row['Value Added (billion won)'] = va_1610 + va_4506
    
    # Sum job creation from both sectors (person/billion won for IO table)
    job_1610 = data["jobcoeff_1610_iotable_2020"][year].sum() if year in data["jobcoeff_1610_iotable_2020"].columns else 0
    job_4506 = data["jobcoeff_4506_iotable_2020"][year].sum() if year in data["jobcoeff_4506_iotable_2020"].columns else 0
    row['Job Creation (person/billion won)'] = job_1610 + job_4506
    
    # Sum direct employment from both sectors
    emp_1610 = data["directemploycoeff_1610_iotable_2020"][year].sum() if year in data["directemploycoeff_1610_iotable_2020"].columns else 0
    emp_4506 = data["directemploycoeff_4506_iotable_2020"][year].sum() if year in data["directemploycoeff_4506_iotable_2020"].columns else 0
    row['Direct Employment (person/billion won)'] = emp_1610 + emp_4506
    
    integrated_io_rows.append(row)

table_integrated_io = pd.DataFrame(integrated_io_rows).set_index('Year')
display(table_integrated_io)

# Table 6: Integrated H2S & H2T
print("\n6. INTEGRATED H2S & H2T (Hydrogen Table)")
print("-"*80)
integrated_h2_rows = []

for year in years:
    row = {'Year': year}
    
    # Sum production from both scenarios
    prod_h2s = data["productioncoeff_H2S_hydrogentable_2020"][year].sum() / 1000 if year in data["productioncoeff_H2S_hydrogentable_2020"].columns else 0
    prod_h2t = data["productioncoeff_H2T_hydrogentable_2020"][year].sum() / 1000 if year in data["productioncoeff_H2T_hydrogentable_2020"].columns else 0
    row['Indirect Production (billion won)'] = prod_h2s + prod_h2t
    
    # No import for hydrogen
    row['Import (billion won)'] = 'N/A'
    
    # Sum value added from both scenarios
    va_h2s = data["valueaddedcoeff_H2S_hydrogentable_2020"][year].sum() / 1000 if year in data["valueaddedcoeff_H2S_hydrogentable_2020"].columns else 0
    va_h2t = data["valueaddedcoeff_H2T_hydrogentable_2020"][year].sum() / 1000 if year in data["valueaddedcoeff_H2T_hydrogentable_2020"].columns else 0
    row['Value Added (billion won)'] = va_h2s + va_h2t
    
    # Sum job creation from both scenarios (billion won for hydrogen)
    job_h2s = data["jobcoeff_H2S_hydrogentable_2020"][year].sum() / 1000 if year in data["jobcoeff_H2S_hydrogentable_2020"].columns else 0
    job_h2t = data["jobcoeff_H2T_hydrogentable_2020"][year].sum() / 1000 if year in data["jobcoeff_H2T_hydrogentable_2020"].columns else 0
    row['Job Creation (billion won)'] = job_h2s + job_h2t
    
    # Sum direct employment from both scenarios
    emp_h2s = data["directemploycoeff_H2S_hydrogentable_2020"][year].sum() if year in data["directemploycoeff_H2S_hydrogentable_2020"].columns else 0
    emp_h2t = data["directemploycoeff_H2T_hydrogentable_2020"][year].sum() if year in data["directemploycoeff_H2T_hydrogentable_2020"].columns else 0
    row['Direct Employment (person/billion won)'] = emp_h2s + emp_h2t
    
    integrated_h2_rows.append(row)

table_integrated_h2 = pd.DataFrame(integrated_h2_rows).set_index('Year')
display(table_integrated_h2)

print("\n" + "="*80)
print("All summary tables created successfully!")
print("="*80)

In [ ]:
# Export all summary tables to one Excel file
output_file = 'output/summary_tables_2030_2040_2050.xlsx'

print(f"Exporting summary tables to {output_file}...")

# Create Excel writer object
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    
    # Sheet 1: Sector 1610
    table_1610.to_excel(writer, sheet_name='1610_IO_Table')
    
    # Sheet 2: Sector 4506
    table_4506.to_excel(writer, sheet_name='4506_IO_Table')
    
    # Sheet 3: H2S
    table_h2s.to_excel(writer, sheet_name='H2S_Hydrogen')
    
    # Sheet 4: H2T
    table_h2t.to_excel(writer, sheet_name='H2T_Hydrogen')
    
    # Sheet 5: Integrated 1610 & 4506
    table_integrated_io.to_excel(writer, sheet_name='Integrated_1610_4506')
    
    # Sheet 6: Integrated H2S & H2T
    table_integrated_h2.to_excel(writer, sheet_name='Integrated_H2S_H2T')

print(f"✅ Successfully exported all 6 summary tables to {output_file}")
print(f"\nSheets created:")
print("  1. 1610_IO_Table")
print("  2. 4506_IO_Table")
print("  3. H2S_Hydrogen")
print("  4. H2T_Hydrogen")
print("  5. Integrated_1610_4506")
print("  6. Integrated_H2S_H2T")

## Next Steps

Add your custom visualizations and analysis below this cell.

In [ ]:
key_map = {
    ('1610', 'indirect_prod'): 'indirect_prod_1610_iotable_2020',
    ('1610', 'indirect_import'): 'indirect_import_1610_iotable_2020',
    ('1610', 'jobcoeff'): 'jobcoeff_1610_iotable_2020',
    ('4506', 'indirect_prod'): 'indirect_prod_4506_iotable_2020',
    ('4506', 'indirect_import'): 'indirect_import_4506_iotable_2020',
    ('4506', 'jobcoeff'): 'jobcoeff_4506_iotable_2020',
    ('H2S', 'indirect_prod'): 'productioncoeff_H2S_hydrogentable_2020',
    ('H2S', 'jobcoeff'): 'jobcoeff_H2S_hydrogentable_2020',
    ('H2T', 'indirect_prod'): 'productioncoeff_H2T_hydrogentable_2020',
    ('H2T', 'jobcoeff'): 'jobcoeff_H2T_hydrogentable_2020',
}
if scenario == '1610&4506':
    
    # 1. effect_type에 따라 사용할 데이터 키(key)를 명확하게 정의
    if effect_type == 'indirect_prod':
        key1 = 'indirect_prod_1610_iotable_2020'
        key2 = 'indirect_prod_4506_iotable_2020'
    elif effect_type == 'indirect_import':
        key1 = 'indirect_import_1610_iotable_2020'
        key2 = 'indirect_import_4506_iotable_2020'
    elif effect_type == 'jobcoeff':
        key1 = 'jobcoeff_1610_iotable_2020'
        key2 = 'jobcoeff_4506_iotable_2020'
    elif effect_type == 'value_added':
        # [!] 이 키 이름이 data 딕셔너리에 있는지 확인이 필요합니다.
        key1 = 'value_added_1610_iotable_2020' 
        key2 = 'value_added_4506_iotable_2020'
    else:
        # 혹시 모를 다른 effect_type에 대한 예외 처리
        raise ValueError(f"'{effect_type}'은(는) 처리할 수 없는 effect_type 입니다.")

    # 2. 필요한 컬럼만 (키 + 선택한 연도) 가져와서 두 개의 데이터프레임을 로드
    # (year 변수에는 '2020'과 같은 단일 연도 문자열이 들어있다고 가정)
    df_1_data = data[key1][['Sector_Code', 'Sector_Name', year]].copy()
    df_2_data = data[key2][['Sector_Code', 'Sector_Name', year]].copy()

    # 3. 'Sector_Code'와 'Sector_Name'을 기준으로 두 데이터를 병합(merge)
    merged_df = pd.merge(
        df_1_data,
        df_2_data,
        on=['Sector_Code', 'Sector_Name'], # 이 컬럼들을 기준으로 합칩니다.
        how='outer',
        suffixes=('_1', '_2') # 겹치는 'year' 컬럼이 '2020_1', '2020_2'가 됨
    )

    # 4. 병합으로 생긴 _1, _2 컬럼 이름을 만듭니다.
    year_1_col = f'{year}_1' # 'year'가 '2020'이면 '2020_1'
    year_2_col = f'{year}_2' # 'year'가 '2020'이면 '2020_2'

    # 5. 'outer' 병합으로 인해 한쪽에만 데이터가 있어 NaN이 된 경우, 0으로 채웁니다.
    merged_df[year_1_col] = merged_df[year_1_col].fillna(0)
    merged_df[year_2_col] = merged_df[year_2_col].fillna(0)

    # 6. 두 컬럼을 더해서, 최종 결과 컬럼 (e.g., '2020')에 합산된 값을 저장합니다.
    merged_df[year] = merged_df[year_1_col] + merged_df[year_2_col]

    # 7. 최종 결과(df1)로 필요한 컬럼만 남깁니다.
    df1 = merged_df[['Sector_Code', 'Sector_Name', year]]


In [ ]:
def get_top_10_sectors(data, scenario, effect_type, year='2050'):
    """
    Get top 10 sectors for a given scenario and effect type.
    
    Parameters:
    - data: Dictionary of loaded CSV dataframes
    - scenario: '1610', '4506', 'H2S', 'H2T', 'H2S&H2T', '1610&4506'
    - effect_type: 'indirect_prod', 'indirect_import', 'jobcoeff'
    - year: Year to analyze (default '2050')
    
    Returns:
    - DataFrame with top 10 sectors
    """
    
    # Map scenario and effect type to data keys
    key_map = {
        ('1610', 'indirect_prod'): 'indirect_prod_1610_iotable_2020',
        ('1610', 'indirect_import'): 'indirect_import_1610_iotable_2020',
        ('1610', 'jobcoeff'): 'jobcoeff_1610_iotable_2020',
        ('4506', 'indirect_prod'): 'indirect_prod_4506_iotable_2020',
        ('4506', 'indirect_import'): 'indirect_import_4506_iotable_2020',
        ('4506', 'jobcoeff'): 'jobcoeff_4506_iotable_2020',
        ('H2S', 'indirect_prod'): 'productioncoeff_H2S_hydrogentable_2020',
        ('H2S', 'jobcoeff'): 'jobcoeff_H2S_hydrogentable_2020',
        ('H2T', 'indirect_prod'): 'productioncoeff_H2T_hydrogentable_2020',
        ('H2T', 'jobcoeff'): 'jobcoeff_H2T_hydrogentable_2020',
    }
    # Handle integrated scenarios
    if scenario == '1610&4506':
        if effect_type == 'indirect_prod':
            df1 = data['indirect_prod_1610_iotable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df2 = data['indirect_prod_4506_iotable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df1['Impact'] = df1[year] + df2[year]
        elif effect_type == 'indirect_import':
            df1 = data['indirect_import_1610_iotable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df2 = data['indirect_import_4506_iotable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df1['Impact'] = df1[year] + df2[year]
        else:  # jobcoeff
            df1 = data['jobcoeff_1610_iotable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df2 = data['jobcoeff_4506_iotable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df1['Impact'] = df1[year] + df2[year]
        
        df_result = df1[['Sector_Code', 'Sector_Name', 'Impact']].copy()
        
    elif scenario == 'H2S&H2T':
        if effect_type == 'indirect_prod':
            df1 = data['productioncoeff_H2S_hydrogentable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df2 = data['productioncoeff_H2T_hydrogentable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df1['Impact'] = df1[year] + df2[year]
        else:  # jobcoeff (no import for hydrogen)
            df1 = data['jobcoeff_H2S_hydrogentable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df2 = data['jobcoeff_H2T_hydrogentable_2020'][['Sector_Code', 'Sector_Name', year]].copy()
            df1['Impact'] = df1[year] + df2[year]
        
        df_result = df1[['Sector_Code', 'Sector_Name', 'Impact']].copy()
        
    else:
        # Single scenario
        if (scenario, effect_type) not in key_map:
            return None  # Invalid combination (e.g., H2S + import)
        
        key = key_map[(scenario, effect_type)]
        if key not in data or year not in data[key].columns:
            return None
        
        df_result = data[key][['Sector_Code', 'Sector_Name', year]].copy()
        df_result = df_result.rename(columns={year: 'Impact'})
    
    # Get top 10 by absolute value
    df_result['abs_impact'] = df_result['Impact'].abs()
    top_10 = df_result.nlargest(10, 'abs_impact')
    
    return top_10[['Sector_Code', 'Sector_Name', 'Impact']].reset_index(drop=True)


def plot_top_10_sectors(data, scenario, effect_type, year='2050'):
    """Create a Plotly bar chart for top 10 sectors."""
    
    top_10 = get_top_10_sectors(data, scenario, effect_type, year)
    
    if top_10 is None or len(top_10) == 0:
        print(f"No data available for {scenario} - {effect_type}")
        return None
    
    # Create color based on positive/negative
    colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_10['Impact']]
    
    # Effect type labels
    effect_labels = {
        'indirect_prod': 'Indirect Production',
        'indirect_import': 'Import',
        'jobcoeff': 'Job Creation'
    }
    
    # Create figure
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=top_10['Sector_Name'],
        x=top_10['Impact'],
        orientation='h',
        marker=dict(color=colors),
        text=top_10['Impact'].apply(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Code: ' + top_10['Sector_Code'].astype(str) + '<br>Impact: %{x:,.0f}<extra></extra>'
    ))
    
    # Determine unit based on effect type and scenario
    if effect_type == 'jobcoeff':
        if scenario in ['H2S', 'H2T', 'H2S&H2T']:
            unit = 'Million Won'
        else:
            unit = 'Person/Billion Won'
    else:
        unit = 'Million Won'
    
    fig.update_layout(
        title=f'Top 10 Sectors - {scenario} | {effect_labels[effect_type]} ({year})',
        xaxis_title=f'Impact ({unit})',
        yaxis_title='',
        height=600,
        showlegend=False,
        template='plotly_white'
    )
    
    fig.show()
    
    return top_10


# Create comprehensive top 10 analysis for all requested combinations
print("=" * 80)
print("TOP 10 SECTORS ANALYSIS - COMPREHENSIVE VIEW")
print("=" * 80)

# Define all combinations to analyze
analysis_combinations = [
    # Sector 1610
    ('1610', 'indirect_prod', '2050'),
    ('1610', 'indirect_import', '2050'),
    ('1610', 'jobcoeff', '2050'),
    
    # Sector 4506
    ('4506', 'indirect_prod', '2050'),
    ('4506', 'indirect_import', '2050'),
    ('4506', 'jobcoeff', '2050'),
    
    # H2S & H2T
    ('H2S&H2T', 'indirect_prod', '2050'),
    ('H2S&H2T', 'jobcoeff', '2050'),
    
    # Integrated 1610 & 4506
    ('1610&4506', 'indirect_prod', '2050'),
    ('1610&4506', 'indirect_import', '2050'),
    ('1610&4506', 'jobcoeff', '2050'),
]

for scenario, effect, year in analysis_combinations:
    print(f"\n{'='*80}")
    print(f"Scenario: {scenario} | Effect: {effect} | Year: {year}")
    print('='*80)
    
    df_top10 = plot_top_10_sectors(data, scenario, effect, year)
    
    if df_top10 is not None:
        print("\nTop 10 Data Table:")
        display(df_top10)

In [ ]:
def create_hydrogen_yearly_trends(data, effect_type, scenarios=['H2S', 'H2T', 'H2S&H2T'], title_suffix=''):
    """
    Create line graphs showing yearly trends for hydrogen scenarios.
    
    Parameters:
    - data: Dictionary of loaded CSV dataframes
    - effect_type: 'production', 'value_added', 'jobcoeff', 'directemploycoeff'
    - scenarios: List of scenarios to plot
    - title_suffix: Additional text for the title
    """
    
    # Map effect types to data keys for hydrogen
    key_map = {
        ('H2S', 'production'): 'productioncoeff_H2S_hydrogentable_2020',
        ('H2S', 'value_added'): 'valueaddedcoeff_H2S_hydrogentable_2020',
        ('H2S', 'jobcoeff'): 'jobcoeff_H2S_hydrogentable_2020',
        ('H2S', 'directemploycoeff'): 'directemploycoeff_H2S_hydrogentable_2020',
        ('H2T', 'production'): 'productioncoeff_H2T_hydrogentable_2020',
        ('H2T', 'value_added'): 'valueaddedcoeff_H2T_hydrogentable_2020',
        ('H2T', 'jobcoeff'): 'jobcoeff_H2T_hydrogentable_2020',
        ('H2T', 'directemploycoeff'): 'directemploycoeff_H2T_hydrogentable_2020',
    }
    
    # Effect type labels and units
    effect_info = {
        'production': {'label': 'Indirect Production', 'unit': 'Billion Won'},
        'value_added': {'label': 'Value Added', 'unit': 'Billion Won'},
        'jobcoeff': {'label': 'Job Creation', 'unit': 'Billion Won'},  # For hydrogen, job is in won
        'directemploycoeff': {'label': 'Direct Employment', 'unit': 'Person/Billion Won'}
    }
    
    fig = go.Figure()
    
    for scenario in scenarios:
        if scenario == 'H2S&H2T':
            # Integrated hydrogen scenario - sum both
            key1 = key_map[('H2S', effect_type)]
            key2 = key_map[('H2T', effect_type)]
            
            if key1 in data and key2 in data:
                df1 = data[key1]
                df2 = data[key2]
                
                year_cols = [col for col in df1.columns if col.isdigit()]
                years = sorted([int(y) for y in year_cols])
                
                # Calculate total for each year (sum of all sectors from both scenarios)
                values = []
                for year in years:
                    year_str = str(year)
                    # Convert to billion won for economic effects
                    if effect_type in ['production', 'value_added', 'jobcoeff']:
                        val1 = df1[year_str].sum() / 1000
                        val2 = df2[year_str].sum() / 1000
                    else:  # directemploycoeff
                        val1 = df1[year_str].sum()
                        val2 = df2[year_str].sum()
                    values.append(val1 + val2)
                
                fig.add_trace(go.Scatter(
                    x=years,
                    y=values,
                    mode='lines+markers',
                    name='H2S & H2T',
                    line=dict(width=3),
                    marker=dict(size=8)
                ))
        else:
            # Single hydrogen scenario
            key = key_map[(scenario, effect_type)]
            
            if key in data:
                df = data[key]
                year_cols = [col for col in df.columns if col.isdigit()]
                years = sorted([int(y) for y in year_cols])
                
                # Calculate total for each year
                if effect_type in ['production', 'value_added', 'jobcoeff']:
                    # Convert to billion won
                    values = [df[str(year)].sum() / 1000 for year in years]
                else:  # directemploycoeff
                    # Keep as is (person/billion won)
                    values = [df[str(year)].sum() for year in years]
                
                fig.add_trace(go.Scatter(
                    x=years,
                    y=values,
                    mode='lines+markers',
                    name=scenario,
                    line=dict(width=3),
                    marker=dict(size=8)
                ))
    
    fig.update_layout(
        title=f'{effect_info[effect_type]["label"]} - Hydrogen Scenarios Yearly Trends{title_suffix}',
        xaxis_title='Year',
        yaxis_title=f'{effect_info[effect_type]["label"]} ({effect_info[effect_type]["unit"]})',
        height=600,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )
    
    fig.show()
    
    return fig


# Create all hydrogen scenario visualizations
print("=" * 80)
print("YEARLY TRENDS ANALYSIS - HYDROGEN SCENARIOS")
print("=" * 80)

# 1. Indirect Production (Hydrogen)
print("\n1. Indirect Production Trends (Hydrogen)")
fig_h2_prod = create_hydrogen_yearly_trends(data, 'production', scenarios=['H2S', 'H2T', 'H2S&H2T'])

# 2. Value Added (Hydrogen)
print("\n2. Value Added Trends (Hydrogen)")
fig_h2_va = create_hydrogen_yearly_trends(data, 'value_added', scenarios=['H2S', 'H2T', 'H2S&H2T'])

# 3. Job Creation (Hydrogen) - Note: in billion won for hydrogen
print("\n3. Job Creation Trends (Hydrogen)")
fig_h2_job = create_hydrogen_yearly_trends(data, 'jobcoeff', scenarios=['H2S', 'H2T', 'H2S&H2T'])

# 4. Direct Employment (Hydrogen)
print("\n4. Direct Employment Trends (Hydrogen)")
fig_h2_emp = create_hydrogen_yearly_trends(data, 'directemploycoeff', scenarios=['H2S', 'H2T', 'H2S&H2T'])

print("\n" + "=" * 80)
print("All hydrogen yearly trends visualizations created!")
print("=" * 80)

In [ ]:
# Export hydrogen yearly trends to HTML files
print("Exporting hydrogen yearly trends to HTML files...")
print("=" * 80)

h2_trends_output_dir = 'output/plotly_charts/yearly_trends_hydrogen'
os.makedirs(h2_trends_output_dir, exist_ok=True)

# Create and export each hydrogen trend
h2_effect_types = ['production', 'value_added', 'jobcoeff', 'directemploycoeff']
h2_effect_labels = {
    'production': 'indirect_production',
    'value_added': 'value_added',
    'jobcoeff': 'job_creation',
    'directemploycoeff': 'direct_employment'
}

for effect in h2_effect_types:
    fig = create_hydrogen_yearly_trends(data, effect, scenarios=['H2S', 'H2T', 'H2S&H2T'], title_suffix='')
    filename = f"{h2_trends_output_dir}/hydrogen_yearly_trends_{h2_effect_labels[effect]}.html"
    fig.write_html(filename)
    print(f"✅ Exported: {filename}")

print("\n" + "=" * 80)
print(f"All hydrogen yearly trends exported to: {h2_trends_output_dir}/")
print("=" * 80)

## Yearly Trends Analysis - Hydrogen Scenarios

Time series analysis for hydrogen scenarios (H2S, H2T, and integrated H2S&H2T).

In [ ]:
def create_yearly_trends(data, effect_type, scenarios=['1610', '4506', '1610&4506'], title_suffix=''):
    """
    Create line graphs showing yearly trends for multiple scenarios.
    
    Parameters:
    - data: Dictionary of loaded CSV dataframes
    - effect_type: 'indirect_prod', 'indirect_import', 'value_added', 'jobcoeff', 'directemploycoeff'
    - scenarios: List of scenarios to plot
    - title_suffix: Additional text for the title
    """
    
    # Map effect types to data keys
    key_map = {
        ('1610', 'indirect_prod'): 'indirect_prod_1610_iotable_2020',
        ('1610', 'indirect_import'): 'indirect_import_1610_iotable_2020',
        ('1610', 'value_added'): 'value_added_1610_iotable_2020',
        ('1610', 'jobcoeff'): 'jobcoeff_1610_iotable_2020',
        ('1610', 'directemploycoeff'): 'directemploycoeff_1610_iotable_2020',
        ('4506', 'indirect_prod'): 'indirect_prod_4506_iotable_2020',
        ('4506', 'indirect_import'): 'indirect_import_4506_iotable_2020',
        ('4506', 'value_added'): 'value_added_4506_iotable_2020',
        ('4506', 'jobcoeff'): 'jobcoeff_4506_iotable_2020',
        ('4506', 'directemploycoeff'): 'directemploycoeff_4506_iotable_2020',
    }
    
    # Effect type labels and units
    effect_info = {
        'indirect_prod': {'label': 'Indirect Production', 'unit': 'Billion Won'},
        'indirect_import': {'label': 'Import', 'unit': 'Billion Won'},
        'value_added': {'label': 'Value Added', 'unit': 'Billion Won'},
        'jobcoeff': {'label': 'Job Creation', 'unit': 'Person/Billion Won'},
        'directemploycoeff': {'label': 'Direct Employment', 'unit': 'Person/Billion Won'}
    }
    
    fig = go.Figure()
    
    for scenario in scenarios:
        if scenario == '1610&4506':
            # Integrated scenario - sum both
            key1 = key_map[('1610', effect_type)]
            key2 = key_map[('4506', effect_type)]
            
            if key1 in data and key2 in data:
                df1 = data[key1]
                df2 = data[key2]
                
                year_cols = [col for col in df1.columns if col.isdigit()]
                years = sorted([int(y) for y in year_cols])
                
                # Calculate total for each year (sum of all sectors from both scenarios)
                values = []
                for year in years:
                    year_str = str(year)
                    val1 = df1[year_str].sum() / 1000 if effect_type in ['indirect_prod', 'indirect_import', 'value_added'] else df1[year_str].sum()
                    val2 = df2[year_str].sum() / 1000 if effect_type in ['indirect_prod', 'indirect_import', 'value_added'] else df2[year_str].sum()
                    values.append(val1 + val2)
                
                fig.add_trace(go.Scatter(
                    x=years,
                    y=values,
                    mode='lines+markers',
                    name='1610 & 4506',
                    line=dict(width=3),
                    marker=dict(size=8)
                ))
        else:
            # Single scenario
            key = key_map[(scenario, effect_type)]
            
            if key in data:
                df = data[key]
                year_cols = [col for col in df.columns if col.isdigit()]
                years = sorted([int(y) for y in year_cols])
                
                # Calculate total for each year
                if effect_type in ['indirect_prod', 'indirect_import', 'value_added']:
                    # Convert to billion won
                    values = [df[str(year)].sum() / 1000 for year in years]
                else:
                    # Keep as is (person/billion won)
                    values = [df[str(year)].sum() for year in years]
                
                fig.add_trace(go.Scatter(
                    x=years,
                    y=values,
                    mode='lines+markers',
                    name=f'Sector {scenario}',
                    line=dict(width=3),
                    marker=dict(size=8)
                ))
    
    fig.update_layout(
        title=f'{effect_info[effect_type]["label"]} - Yearly Trends{title_suffix}',
        xaxis_title='Year',
        yaxis_title=f'{effect_info[effect_type]["label"]} ({effect_info[effect_type]["unit"]})',
        height=600,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )
    
    fig.show()
    
    return fig


# Create all the requested visualizations
print("=" * 80)
print("YEARLY TRENDS ANALYSIS - ECONOMIC EFFECTS")
print("=" * 80)

# 1. Indirect Production
print("\n1. Indirect Production Trends")
fig1 = create_yearly_trends(data, 'indirect_prod', scenarios=['1610', '4506', '1610&4506'])

# 2. Import
print("\n2. Import Trends")
fig2 = create_yearly_trends(data, 'indirect_import', scenarios=['1610', '4506', '1610&4506'])

# 3. Value Added
print("\n3. Value Added Trends")
fig3 = create_yearly_trends(data, 'value_added', scenarios=['1610', '4506', '1610&4506'])

# 4. Job Creation
print("\n4. Job Creation Trends")
fig4 = create_yearly_trends(data, 'jobcoeff', scenarios=['1610', '4506', '1610&4506'])

# 5. Direct Employment
print("\n5. Direct Employment Trends")
fig5 = create_yearly_trends(data, 'directemploycoeff', scenarios=['1610', '4506', '1610&4506'])

print("\n" + "=" * 80)
print("All yearly trends visualizations created!")
print("=" * 80)

In [ ]:
# Export yearly trends to HTML files
print("Exporting yearly trends to HTML files...")
print("=" * 80)

trends_output_dir = 'output/plotly_charts/yearly_trends'
os.makedirs(trends_output_dir, exist_ok=True)

# Create and export each trend
effect_types = ['indirect_prod', 'indirect_import', 'value_added', 'jobcoeff', 'directemploycoeff']
effect_labels = {
    'indirect_prod': 'indirect_production',
    'indirect_import': 'import',
    'value_added': 'value_added',
    'jobcoeff': 'job_creation',
    'directemploycoeff': 'direct_employment'
}

for effect in effect_types:
    fig = create_yearly_trends(data, effect, scenarios=['1610', '4506', '1610&4506'], title_suffix='')
    filename = f"{trends_output_dir}/yearly_trends_{effect_labels[effect]}.html"
    fig.write_html(filename)
    print(f"✅ Exported: {filename}")

print("\n" + "=" * 80)
print(f"All yearly trends exported to: {trends_output_dir}/")
print("=" * 80)

## Yearly Trends Analysis - Multiple Line Graphs

Time series analysis showing economic effects across all years for different scenarios.

In [ ]:
# Option 2: Export charts to static images (PNG/PDF)
# Good for presentations, reports, or emails
# Note: Requires kaleido package. Install with: pip install kaleido

print("Exporting charts to PNG images...")
print("=" * 80)

# Uncomment the following lines to export to PNG:
# for scenario, effect, year in export_combinations:
#     top_10 = get_top_10_sectors(data, scenario, effect, year)
#     
#     if top_10 is None or len(top_10) == 0:
#         continue
#     
#     # Create figure (same as above)
#     colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_10['Impact']]
#     effect_labels = {
#         'indirect_prod': 'Indirect Production',
#         'indirect_import': 'Import',
#         'jobcoeff': 'Job Creation'
#     }
#     
#     fig = go.Figure()
#     fig.add_trace(go.Bar(
#         y=top_10['Sector_Name'],
#         x=top_10['Impact'],
#         orientation='h',
#         marker=dict(color=colors),
#         text=top_10['Impact'].apply(lambda x: f'{x:,.0f}'),
#         textposition='outside'
#     ))
#     
#     if effect == 'jobcoeff':
#         unit = 'Million Won' if scenario in ['H2S', 'H2T', 'H2S&H2T'] else 'Person/Billion Won'
#     else:
#         unit = 'Million Won'
#     
#     fig.update_layout(
#         title=f'Top 10 Sectors - {scenario} | {effect_labels[effect]} ({year})',
#         xaxis_title=f'Impact ({unit})',
#         height=600,
#         template='plotly_white'
#     )
#     
#     # Export to PNG
#     png_filename = f"{output_dir}/top10_{scenario}_{effect}_{year}.png"
#     fig.write_image(png_filename, width=1200, height=600)
#     print(f"✅ Exported PNG: {png_filename}")

print("\n💡 To enable PNG export, install kaleido:")
print("   pip install kaleido")
print("   Then uncomment the code above and run the cell again.")

In [ ]:
# Option 1: Export individual charts to interactive HTML files
# Your colleague can open these in any web browser and interact with them

import os
output_dir = 'output/plotly_charts'
os.makedirs(output_dir, exist_ok=True)

print("Exporting Plotly charts to HTML files...")
print("=" * 80)

# Export all combinations
export_combinations = [
    ('1610', 'indirect_prod', '2050'),
    ('1610', 'indirect_import', '2050'),
    ('1610', 'jobcoeff', '2050'),
    ('4506', 'indirect_prod', '2050'),
    ('4506', 'indirect_import', '2050'),
    ('4506', 'jobcoeff', '2050'),
    ('H2S&H2T', 'indirect_prod', '2050'),
    ('H2S&H2T', 'jobcoeff', '2050'),
    ('1610&4506', 'indirect_prod', '2050'),
    ('1610&4506', 'indirect_import', '2050'),
    ('1610&4506', 'jobcoeff', '2050'),
]

for scenario, effect, year in export_combinations:
    top_10 = get_top_10_sectors(data, scenario, effect, year)
    
    if top_10 is None or len(top_10) == 0:
        continue
    
    # Create the chart
    colors = ['#2ecc71' if x > 0 else '#e74c3c' for x in top_10['Impact']]
    
    effect_labels = {
        'indirect_prod': 'Indirect Production',
        'indirect_import': 'Import',
        'jobcoeff': 'Job Creation'
    }
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=top_10['Sector_Name'],
        x=top_10['Impact'],
        orientation='h',
        marker=dict(color=colors),
        text=top_10['Impact'].apply(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Code: ' + top_10['Sector_Code'].astype(str) + '<br>Impact: %{x:,.0f}<extra></extra>'
    ))
    
    if effect == 'jobcoeff':
        if scenario in ['H2S', 'H2T', 'H2S&H2T']:
            unit = 'Million Won'
        else:
            unit = 'Person/Billion Won'
    else:
        unit = 'Million Won'
    
    fig.update_layout(
        title=f'Top 10 Sectors - {scenario} | {effect_labels[effect]} ({year})',
        xaxis_title=f'Impact ({unit})',
        yaxis_title='',
        height=600,
        showlegend=False,
        template='plotly_white'
    )
    
    # Export to HTML
    filename = f"{output_dir}/top10_{scenario}_{effect}_{year}.html"
    fig.write_html(filename)
    print(f"✅ Exported: {filename}")

print("\n" + "=" * 80)
print(f"All charts exported to: {output_dir}/")
print("Your colleague can open these HTML files in any web browser!")
print("=" * 80)

## Export Plotly Charts

Multiple options to share your visualizations with colleagues.

## Top 10 Sectors Analysis

Interactive visualizations showing top 10 affected sectors for different scenarios and effect types.

## Categorical Heatmaps with code_h Categories

Creating heatmaps aggregated by code_h categories showing impacts over years for integrated scenarios.

In [ ]:
from codecs import utf_8_decode
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

# --- 1. 데이터 불러오기 및 기준 값 설정 ---
file_path = 'indirect_prod_sample.xlsx'
df= pd.read_excel(file_path, sheet_name='Sector_Impacts_by_Year')

# ***
# 가정: 이전 요청에 따라 'Year_2030'을 기준으로 정렬하고 색상을 지정합니다.
# 만약 다른 연도나 'Total_Impact'를 사용하고 싶다면, 이 변수를 수정하세요.
# ***
value_column = 'Year_2030'

# 시각화에 필요한 컬럼만 추출
df_prep = df[['Sector_Name', 'Code_H', value_column]].copy()

# --- 2. X축(Code_H) 기준으로 데이터 재구성 ---

# X축이 될 고유한 Code_H 리스트 (정렬)
all_codes = sorted(df_prep['Code_H'].unique())

# 각 Code_H 별로 정렬된 'Sector_Name'과 '값'을 저장할 딕셔너리
sorted_data = {}
# 히트맵의 Y축 길이(최대 랭킹)를 찾기 위한 변수
max_len = 0 

for code in all_codes:
    # 현재 Code_H에 해당하는 데이터만 필터링
    df_group = df_prep[df_prep['Code_H'] == code]
    
    # 기준 값(Year_2030)으로 내림차순 정렬 (Y축: 1등 -> 꼴등)
    df_group_sorted = df_group.sort_values(by=value_column, ascending=False)
    
    # 정렬된 Sector_Name 리스트와 값 리스트를 저장
    sorted_data[code] = {
        'labels': df_group_sorted['Sector_Name'].tolist(),
        'values': df_group_sorted[value_column].tolist()
    }
    
    # 가장 긴 그룹(가장 많은 섹터를 가진)의 길이를 추적
    if len(df_group_sorted) > max_len:
        max_len = len(df_group_sorted)

# --- 3. 히트맵용 데이터프레임 생성 (Padding) ---

# 이 작업은 각 Code_H 그룹의 섹터 수가 다르기 때문에 필요합니다.
# 모든 열(Code_H)이 동일한 Y축 길이(max_len)를 갖도록 맞춰줍니다.

# 1. 셀 텍스트(Sector_Name)용 데이터프레임
df_labels = pd.DataFrame(index=range(max_len), columns=all_codes)
# 2. 셀 색상(값)용 데이터프레임
df_values = pd.DataFrame(index=range(max_len), columns=all_codes, dtype=float)

# 딕셔너리에 저장된 정렬된 리스트를 데이터프레임에 채워넣기
for code in all_codes:
    labels = sorted_data[code]['labels']
    values = sorted_data[code]['values']
    
    # max_len보다 짧은 리스트는 빈 값으로 채워줍니다.
    # (라벨: 빈 문자열 '', 값: np.nan - 색상 없음)
    padded_labels = labels + [''] * (max_len - len(labels))
    padded_values = values + [np.nan] * (max_len - len(values))
    
    df_labels[code] = padded_labels
    df_values[code] = padded_values

# --- 4. 시각화 ---

# 사용할 폰트 경로 설정 (예시: 애플고딕, 환경에 따라 수정)
font_path = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'  # Mac 예시
font_properties = fm.FontProperties(fname=font_path)

# 플롯의 크기를 동적으로 설정 (매우 길어야 함)
fig_height = max(60, max_len * 0.5) # Y축 높이
fig_width = max(20, len(all_codes) * 1.5) # X축 너비

plt.figure(figsize=(fig_width, fig_height))

# sns.heatmap의 annot_kws에는 직접 fontproperties를 지정할 수 없어 
# 아래에서 축, 타이틀, 컬러바 등에 명시적으로 fontproperties를 적용

ax = sns.heatmap(
    df_values,          # 5. 셀 색상은 이 숫자 데이터를 따름
    annot=df_labels,    # 4. 셀 안에는 이 텍스트(Sector_Name)를 표시
    fmt='s',            # 텍스트 포맷 (string)
    cmap='vlag',        # 양수(파랑), 음수(빨강)의 발산형 컬러맵
    center=0,           # 0을 기준으로 색상 매핑
    linewidths=0.5,
    linecolor='lightgray',
    annot_kws={"size": 6, "color": "black", "fontproperties": font_properties}, # 폰트 적용
    cbar_kws={'label': f'Impact Value ({value_column})'}
)

# --- 5. 축 설정 ---

# 2. X축(Code_H) 라벨을 플롯 상단으로 이동
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.tick_params(axis='x', rotation=45) # X축 라벨 45도 회전

# 타이틀, 라벨 폰트 적용
ax.set_title(f'Sector Ranking within each Code_H (Sorted by {value_column} Impact)', fontsize=20, pad=40, fontproperties=font_properties)
ax.set_xlabel('Code_H', fontsize=14, fontproperties=font_properties)
ax.set_ylabel(f'Rank (1 to {max_len})', fontsize=14, fontproperties=font_properties)

# X, Y 틱 라벨 폰트도 변경
for label in ax.get_xticklabels():
    label.set_fontproperties(font_properties)
for label in ax.get_yticklabels():
    label.set_fontproperties(font_properties)

# 컬러바 라벨과 폰트
cbar = ax.collections[0].colorbar
cbar.set_label(f'Impact Value ({value_column})', fontsize=13, fontproperties=font_properties)
for t in cbar.ax.get_yticklabels():
    t.set_fontproperties(font_properties)

# Y축의 숫자 틱(0, 10, 20...)은 제거 (순위는 위에서 아래로 정렬된 것으로 표현됨)
ax.set_yticks([]) 

plt.savefig('code_h_sector_ranking_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()